In [ ]:
input_data = None
input_doc = None
output_data = None
util = None
display_util = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

# from IPython.display import Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rename,
    display_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
)

# spender_postmortem_diagnosen

In this dataset diagnoses results from the potential donor are provided by the {term}`ET` and {term}`DSO` in the file `element_spender_postmortem_diagnosen.csv`.

We perform the common steps outlined in [](../02_preprocessing/index.md). As the {term}`ET` data is not in a longitudinal format like in the {term}`DSO` part of the data and only contains death information, which is also available in the {term}`DSO` data, the {term}`ET` data is removed. This death data is added to every row, otherwise only common translations are applied. 

## Unprocessed input data

In [ ]:
data = pd.read_csv(input_data, sep=";", low_memory=False)
display_data_doc(data=data, official_doc=pd.read_csv(input_doc))

## Technical Steps

For this file the general plan for technical preprocessing was followed.

### Removal of Non-Informative Columns

First empty and duplicated columns were removed (see [](general:ecf)). We do not use the {term}`DSO` identifiers, so the column `SPostmDiagnosenIdDSOKennnummerDSO` was also removed.

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data).drop(columns="SPostmDiagnosenIdDSOKennnummerDSO")

### Renaming of Columns

New names were used for the columns. (see [](general:cr)).

In [ ]:
renaming = {
    "SPostmBasisTodesursacheET": "death_reason_et",
    "SPostmBasisTodesursacheICD10BeschreibungET": "death_reason_icd_et",
    "SPostmBasisTodesursacheICD10ET": "death_reason_icd_code_et",
    "SPostmDiagnosenDateET": "death_date_et",
    "SPostmDiagnosenDiagnoseBehandeltDSO": "treated",
    "SPostmDiagnosenDiagnoseBeschreibungDSO": "diagnosis",
    "SPostmDiagnosenDiagnoseICD10DSO": "diagnosis_code",
    "SPostmDiagnosenDiagnoseICD10SternDSO": "diagnosis_code_additional",
    "SPostmDiagnosenDiagnosezusatzDSO": "diagnosis_additional",
    "SPostmDiagnosenErfahrenAmDateDSO": "communication_date",
    "SPostmDiagnosenErkrankungBeginnDateDSO": "disease_start_date",
    "SPostmDiagnosenErkrankungEndeDateDSO": "disease_end_date",
    "SPostmDiagnosenHirnschaedigungArtDSO": "brain_damage_type",
    "SPostmDiagnosenHirnschaedigungLokalisationDSO": "brain_damage_localisation",
    "SPostmDiagnosenIdSpenderNrETDSO": "donor_et_dso",
    "SPostmDiagnosenIdSpenderNrETET": "donor_et_id_et",
    "SPostmDiagnosenKlassifikationDSO": "diagnosis_type",
    "SPostmDiagnosenTodesdatumDSO": "death_date",
    "SPostmDiagnosenTodesursacheDSO": "death_type",
    "SPostmDiagnosenAusschlussDarmDSO": "colon_excluded",
    "SPostmDiagnosenAusschlussHDSO": "heart_excluded",
    "SPostmDiagnosenAusschlussLeDSO": "liver_excluded",
    "SPostmDiagnosenAusschlussLesplitLinksDSO": "liver_left_excluded",
    "SPostmDiagnosenAusschlussLuLinksDSO": "lung_left_excluded",
    "SPostmDiagnosenAusschlussLuRechtsDSO": "lung_right_excluded",
    "SPostmDiagnosenAusschlussNLinksDSO": "kidney_left_excluded",
    "SPostmDiagnosenAusschlussNRechtsDSO": "kidney_right_excluded",
    "SPostmDiagnosenAusschlussPDSO": "pancreas_excluded",
}


data = rename(data, renaming)
data.to_parquet(output_data)